%md
# Exploratory Data Analysis on the Marathos Dataset Part 3
- Athlete club
- athlete country
- Athlete year of birth
- Athlete gender
- Athlete age category
- Athelete ID

## Load Dataset

In [0]:
# Use the raw dataset inside the volume
VOLUME_PATH = "/Volumes/marathos_catalog/default/raw/"

# Use spark and sql
spark.sql(f"LIST '{VOLUME_PATH}'")

In [0]:
# See data path
DATA_PATH = "/Volumes/marathos_catalog/default/raw/data/"

display(spark.sql(f"LIST '{DATA_PATH}'"))

In [0]:
# Show what is inside the data folder. Using this method for better debugging later.
FILE_PATH = "/Volumes/marathos_catalog/default/raw/data/TWO_CENTURIES_OF_UM_RACES.csv"

# read csv
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(FILE_PATH)
)

display(df)

## Check Schema

In [0]:
# Check the schema that Spark inferred from the CSV file | column name + data type
from pyspark.sql.functions import col, count, when, countDistinct, trim, upper
df.printSchema()

## Athlete Club
### Check NULLS


In [0]:
# Calculate null percentage for Athlete club

total_rows = df.count()

athlete_club_null_percentage = (athlete_club_null_count / total_rows) * 100

print(f"Total rows: {total_rows}")
print(f"Null values in Athlete club: {athlete_club_null_count}")
print(f"Null percentage in Athlete club: {athlete_club_null_percentage:.2f}%")

- The `Athlete club` column has **2,826,373 null values**, which is around **37.88%** of the dataset.
- Since club information is not essential for identifying the event result itself, I will not remove rows only because `Athlete club` is missing.
- **In the Silver layer, I can keep the column and standardize them to a value such as `unknown`.**


### Disticnt Club Values

In [0]:
# Count distinct Athlete club values

distinct_athlete_club_count = (
    df
    .select("Athlete club")
    .distinct()
    .count()
)

print(f"Number of distinct Athlete club values: {distinct_athlete_club_count}")

In [0]:
# Show most common Athlete club values

athlete_club_counts_df = (
    df
    .groupBy("Athlete club")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(athlete_club_counts_df)

In [0]:
# Check formatting patterns in Athlete club

from pyspark.sql.functions import col, count, when

athlete_club_pattern_df = (
    df
    .withColumn(
        "club_pattern_type",
        when(col("Athlete club").isNull(), "null")
        .when(col("Athlete club").rlike(r"^\*"), "starts_with_star")
        .when(col("Athlete club").rlike(r"[^\x00-\x7F]"), "contains_non_ascii_characters")
        .when(col("Athlete club").rlike(r"^\s*$"), "empty_or_whitespace")
        .otherwise("standard_text")
    )
    .groupBy("club_pattern_type")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(athlete_club_pattern_df)

####  Formatting patterns
I inspected formatting patterns in the `Athlete club` column. The column contains:
    - **2,826,373 null values**
    - **2,233,808 values that start with `*`**
    - **2,091,425 standard text values**
    - **309,589 values with non-ASCII characters**

- The high number of null values shows that club information is often missing. However, this column is not essential for identifying race results, so I will not remove rows only because `Athlete club` is null.

- **Values starting with `*` appear to be a raw formatting issue or convention. In the Silver layer, these values can be standardized by removing the leading `*`.**

- Values with non-ASCII characters are not automatically invalid. Since this is an international dataset, club names may contain accents, Cyrillic, Kanji, or other non-Latin characters.

- In the Silver layer, I should apply light text standardization, such as trimming whitespace and removing leading `*`, while preserving valid international characters.

In [0]:
# Inspect Athlete club values that contain non-ASCII characters

non_ascii_athlete_club_df = (
    df
    .filter(
        col("Athlete club").isNotNull()
        & col("Athlete club").rlike(r"[^\x00-\x7F]")
    )
    .groupBy("Athlete club")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(non_ascii_athlete_club_df)

## Athlete Country
### Check country code

In [0]:
# Check unique country codes from the marathon dataset
marathon_country_codes_df = (
    df
    .select(
        upper(trim(col("Athlete country"))).alias("athlete_country_code")
    )
    .distinct()
    .orderBy("athlete_country_code")
)

display(marathon_country_codes_df)

### Check NULLS>

In [0]:
# Check null values in Athlete country

from pyspark.sql.functions import col

athlete_country_null_count = (
    df
    .filter(col("Athlete country").isNull())
    .count()
)

print(f"Null values in Athlete country: {athlete_country_null_count}")

In [0]:
# Inspect rows where Athlete country is null

display(
    df
    .filter(col("Athlete country").isNull())
)

- The `Athlete country` column has **3 null values**.
- Since the dataset contains over 7 million rows, this is a very small number of missing values.
- **In the Silver layer, these rows can be removed.**

In [0]:
# Count distinct Athlete country values

distinct_athlete_country_count = (
    df
    .select("Athlete country")
    .distinct()
    .count()
)

print(f"Number of distinct Athlete country values: {distinct_athlete_country_count}")

In [0]:
# Show most represented athlete countries

athlete_country_counts_df = (
    df
    .groupBy("Athlete country")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(athlete_country_counts_df)

## Athlete Gender
### Check NULLS
- In the Silver layer, these 7 rows are problematic in other columns too. REMOVE

In [0]:
# Check null values in Athlete gender

athlete_gender_null_count = (
    df
    .filter(col("Athlete gender").isNull())
    .count()
)

print(f"Null values in Athlete gender: {athlete_gender_null_count}")

In [0]:
# Count rows by Athlete gender

athlete_gender_counts_df = (
    df
    .groupBy("Athlete gender")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(athlete_gender_counts_df)

## Athlete Year of birth
### Check NULLS

In [0]:
# Check null values in Athlete year of birth

athlete_year_of_birth_null_count = (
    df
    .filter(col("Athlete year of birth").isNull())
    .count()
)

print(f"Null values in Athlete year of birth: {athlete_year_of_birth_null_count}")

In [0]:
# Calculate null percentage for Athlete year of birth

total_rows = df.count()

athlete_year_of_birth_null_percentage = (
    athlete_year_of_birth_null_count / total_rows
) * 100

print(f"Total rows: {total_rows}")
print(f"Null values in Athlete year of birth: {athlete_year_of_birth_null_count}")
print(f"Null percentage in Athlete year of birth: {athlete_year_of_birth_null_percentage:.2f}%")

In [0]:
# Descriptive summary of Athlete year of birth

display(
    df
    .select("Athlete year of birth")
    .summary()
)

- In Silver, I will use Year of event and Athlete year of birth to calculate athlete age. If the birth year is missing, the age will also be missing. If the calculated age is unrealistic, I will mark it as invalid or set it to null. I will not remove the whole row just because birth year is missing, because the race result itself can still be useful.

## Athlete age category
### Check NULLS


In [0]:
# Check null values in Athlete age category

athlete_age_category_null_count = (
    df
    .filter(col("Athlete age category").isNull())
    .count()
)

print(f"Null values in Athlete age category: {athlete_age_category_null_count}")

In [0]:
# Calculate null percentage for Athlete age category

total_rows = df.count()

athlete_age_category_null_percentage = (
    athlete_age_category_null_count / total_rows
) * 100

print(f"Total rows: {total_rows}")
print(f"Null values in Athlete age category: {athlete_age_category_null_count}")
print(f"Null percentage in Athlete age category: {athlete_age_category_null_percentage:.2f}%")

In [0]:
# Count rows by Athlete age category

athlete_age_category_counts_df = (
    df
    .groupBy("Athlete age category")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(athlete_age_category_counts_df)

- In the Silver layer, I will standardize missing age categories as `Unknown`.

## Athlete ID
### Check NULLS

In [0]:
# Check null values in Athlete ID

athlete_id_null_count = (
    df
    .filter(col("Athlete ID").isNull())
    .count()
)

print(f"Null values in Athlete ID: {athlete_id_null_count}")

In [0]:
# Check if Athlete ID is unique per row
total_rows = df.count()

distinct_athlete_id_count = (
    df
    .select(countDistinct("Athlete ID").alias("distinct_count"))
    .collect()[0]["distinct_count"]
)

print(f"Total rows: {total_rows}")
print(f"Distinct Athlete IDs: {distinct_athlete_id_count}")
print(f"Repeated Athlete ID rows: {total_rows - distinct_athlete_id_count}")

- Athlete ID identifies an athlete, not a single race result.
- So one athlete can appear in multiple rows because they may have participated in multiple races/events over time.

#### Dimensional Modelling for athlete
```
dim_athlete
├── athlete_id
├── athlete_gender
├── athlete_year_of_birth
├── athlete_age_category
├── athlete_country
```
race results table can reference it:
```
fct_results
├── result_id
├── athlete_id
├── event_id
├── performance
```
